# Asynchronous Ingestion — Step by Step

This notebook **builds a working asynchronous ingestion pipeline**, one cell
at a time, so you can run it top to bottom and watch each piece come alive:
an object-storage stub, a thread-safe job queue, a background worker pool, a
non-blocking upload call, live status polling, and retry/backoff with
dead-lettering.

It's the hands-on version of Section 5 ("Asynchronous Ingestion Workflow") of
`Multimodal_RAG_Production_Design.docx` — same design, but here you actually
run it instead of just reading about it. Every cell is standard-library
Python only (`queue`, `threading`, `dataclasses`, `time`) — no installs, no
API keys, works identically in Colab or local Jupyter.

**Why this matters for your V1 app:** today, `ui/app.py` calls
`MultimodalDocumentIngestion.ingest_pdf()` directly and the Streamlit request
blocks until parsing/OCR/chunking/embedding all finish. This notebook shows
the fix: the upload call returns immediately, and a separate worker pool
does the slow work in the background while the UI polls a status field.

**Run it:** `Runtime → Run all` (Colab) or `Run → Run All Cells` (Jupyter).
Cells are meant to be read and re-run individually too — each one prints
something you can inspect before moving to the next.

## Step 0 — Imports

Nothing exotic: `queue.Queue` is the job queue, `threading.Thread` is the
worker pool, everything else is bookkeeping.

In [1]:
import queue
import threading
import time
import random
import uuid
import enum
from dataclasses import dataclass, field
from datetime import datetime

import pandas as pd
from IPython.display import display, clear_output

random.seed(11)
print("Ready.")

Ready.


## Step 1 — The status state machine

Same states as the design doc:

```
uploaded -> parsing -> chunking -> indexing -> ready
              |          |          |
              +----------+----------+--> failed --> retrying --> (parsing | dead_letter)
```

We track a timestamped `history` on every job so we can print a timeline
later, not just the current state.

In [2]:
class Status(enum.Enum):
    UPLOADED = "uploaded"
    PARSING = "parsing"
    CHUNKING = "chunking"
    INDEXING = "indexing"
    READY = "ready"
    FAILED = "failed"
    RETRYING = "retrying"
    DEAD_LETTER = "dead_letter"

TERMINAL = {Status.READY, Status.DEAD_LETTER}

@dataclass
class IngestionJob:
    job_id: str
    filename: str
    file_bytes: bytes
    status: Status = Status.UPLOADED
    attempt_count: int = 0
    max_attempts: int = 3
    error_message: str | None = None
    always_fail: bool = False          # simulates a poison-pill / corrupt PDF
    history: list = field(default_factory=list)

    def set_status(self, new_status: Status, error: str | None = None):
        if new_status is not self.status:   # skip no-op self-transitions in the timeline
            self.history.append((datetime.now().strftime("%H:%M:%S.%f")[:-3], self.status.value, new_status.value))
        self.status = new_status
        self.error_message = error

print("State machine defined.")

State machine defined.


## Step 2 — Object Storage and the Metadata store (in-memory stand-ins)

In production these are S3/GCS (raw bytes) and PostgreSQL (job/document
rows). Here they're a `dict` behind a lock — same *role* in the architecture,
just not durable across a process restart. Swapping the dict for a real
bucket/table later doesn't change anything else in this notebook.

In [3]:
_storage_lock = threading.Lock()
OBJECT_STORAGE: dict[str, bytes] = {}     # object_key -> raw bytes
JOBS: dict[str, IngestionJob] = {}        # job_id -> job (stands in for the documents/ingestion_jobs tables)

def put_object(key: str, data: bytes) -> None:
    with _storage_lock:
        OBJECT_STORAGE[key] = data

def get_object(key: str) -> bytes:
    with _storage_lock:
        return OBJECT_STORAGE[key]

print("Object storage + metadata store ready. Objects stored:", len(OBJECT_STORAGE))

Object storage + metadata store ready. Objects stored: 0


## Step 3 — The Job Queue

`queue.Queue` is thread-safe out of the box — this is standing in for
SQS/RabbitMQ/Celery+Redis in production. Workers (Step 4) will block on
`JOB_QUEUE.get()` until something is enqueued.

In [4]:
JOB_QUEUE: queue.Queue[str] = queue.Queue()   # holds job_ids
print("Job queue ready. Depth:", JOB_QUEUE.qsize())

Job queue ready. Depth: 0


## Step 4 — The Document API `upload()` call — non-blocking

This is what a `POST /documents` endpoint does in production: write the raw
bytes to Object Storage, create a job row with `status=uploaded`, enqueue the
job, and **return immediately**. It never calls the parser itself.

We time the call to prove it doesn't block on document size.

In [5]:
def upload_document(filename: str, file_bytes: bytes, always_fail: bool = False) -> str:
    job_id = str(uuid.uuid4())[:8]
    object_key = f"uploads/{job_id}/{filename}"

    put_object(object_key, file_bytes)                      # 1. durable storage
    job = IngestionJob(job_id=job_id, filename=filename, file_bytes=file_bytes, always_fail=always_fail)
    job.set_status(Status.UPLOADED)
    JOBS[job_id] = job                                       # 2. metadata row created

    JOB_QUEUE.put(job_id)                                    # 3. enqueue — this is the ENTIRE handoff to async work
    return job_id                                             # 4. return immediately (HTTP 202 in production)

# Prove it's non-blocking: a "large" 5 MB payload uploads just as fast as a tiny one.
start = time.perf_counter()
demo_id = upload_document("large_contract.pdf", b"x" * 5_000_000)
elapsed_ms = (time.perf_counter() - start) * 1000
print(f"upload_document() returned in {elapsed_ms:.2f} ms for a 5 MB file — parsing hasn't even started yet.")
print("Job status right now:", JOBS[demo_id].status.value)

upload_document() returned in 33.63 ms for a 5 MB file — parsing hasn't even started yet.
Job status right now: uploaded


## Step 5 — The Parser Worker

This is the function each background worker thread runs in a loop: pull a
job off the queue, walk it through `parsing -> chunking -> indexing -> ready`,
and on failure, retry with exponential backoff up to `max_attempts` before
dead-lettering it.

**Where your real code plugs in:** replace the `time.sleep(...)` calls below
with the actual pipeline —

```python
parser = ComplexPDFParser(pdf_path=..., output_dir=...)   # src/parsing.py
parsed = parser.parse(save_output=True)
ingestion = MultimodalDocumentIngestion()                  # src/ingestion.py
ingestion.ingest_documents(parsed["documents"])
```

— everything around it (status transitions, retry, dead-letter, the queue
itself) stays exactly the same.

In [6]:
def process_stage(job: IngestionJob, stage_name: str, duration: float = 0.15):
    """Stands in for one pipeline stage. Raises to simulate a transient failure."""
    time.sleep(duration)
    if job.always_fail:
        raise RuntimeError(f"{stage_name} failed: simulated corrupt/poison-pill document")

def handle_job(job: IngestionJob):
    job.set_status(Status.PARSING)
    process_stage(job, "parsing (ComplexPDFParser: PyMuPDF/pdfplumber/Tesseract)")

    job.set_status(Status.CHUNKING)
    process_stage(job, "chunking (RecursiveCharacterTextSplitter + table/image prep)")

    job.set_status(Status.INDEXING)
    process_stage(job, "indexing (embeddings + Qdrant upsert)")

    job.set_status(Status.READY)

def worker_loop(worker_name: str, stop_event: threading.Event):
    while not stop_event.is_set():
        try:
            job_id = JOB_QUEUE.get(timeout=0.2)
        except queue.Empty:
            continue

        job = JOBS[job_id]
        try:
            handle_job(job)
            print(f"[{worker_name}] {job.filename} ({job.job_id}) -> ready")
        except RuntimeError as exc:
            job.attempt_count += 1
            job.set_status(Status.FAILED, error=str(exc))
            if job.attempt_count >= job.max_attempts:
                job.set_status(Status.RETRYING)
                job.set_status(Status.DEAD_LETTER, error=str(exc))
                print(f"[{worker_name}] {job.filename} ({job.job_id}) -> DEAD LETTER after {job.attempt_count} attempts: {exc}")
            else:
                job.set_status(Status.RETRYING)
                backoff = 0.1 * (2 ** job.attempt_count)   # jittered exponential backoff in production
                print(f"[{worker_name}] {job.filename} ({job.job_id}) failed (attempt {job.attempt_count}), retrying in {backoff:.2f}s: {exc}")
                time.sleep(backoff)
                # Left in RETRYING until a worker actually dequeues it again — handle_job()
                # is what transitions it back to PARSING, so the status board is honest
                # about "waiting to retry" vs. "actively retrying".
                JOB_QUEUE.put(job_id)   # requeue — only the failed stage restarts, not the whole request
        finally:
            JOB_QUEUE.task_done()

print("Worker function defined.")

Worker function defined.


## Step 6 — Start the worker pool

Two workers, running as background daemon threads, pulling from the same
queue independently — this is the "Parser Workers (autoscaled pool)" box in
the architecture diagram. In production this is a pool of container
replicas, not threads, but the behavior — pull job, process, ack — is
identical.

In [7]:
stop_event = threading.Event()
workers = [
    threading.Thread(target=worker_loop, args=(f"worker-{i}", stop_event), daemon=True)
    for i in range(2)
]
for w in workers:
    w.start()

print(f"Started {len(workers)} background workers:", [w.name for w in workers])

Started 2 background workers: ['Thread-5 (worker_loop)', 'Thread-6 (worker_loop)']


## Step 7 — Upload several documents and watch the calls stay instant

Five uploads in a row, each timed. Notice the calls all return in
milliseconds regardless of order — the actual parsing is happening
concurrently on the two worker threads started in Step 6, completely
decoupled from these calls.

In [8]:
docs_to_upload = [
    ("Client_Contracts_Policies_and_Incident_Records.pdf", 1_500_000, False),
    ("llama2-research-paper.pdf", 13_000_000, False),
    ("vendor_msa_amendment.pdf", 800_000, False),
    ("incident_report_q3.pdf", 400_000, False),
    ("support_sla_addendum.pdf", 250_000, False),
]

uploaded_ids = []
for filename, size, fail in docs_to_upload:
    start = time.perf_counter()
    job_id = upload_document(filename, b"x" * size, always_fail=fail)
    elapsed_ms = (time.perf_counter() - start) * 1000
    uploaded_ids.append(job_id)
    print(f"POST /documents  {filename:52s} -> 202 Accepted in {elapsed_ms:5.2f} ms  (job_id={job_id})")

print()
print("All 5 uploads accepted. Processing is now happening in the background —")
print("continue to Step 8 to watch it happen live.")

POST /documents  Client_Contracts_Policies_and_Incident_Records.pdf   -> 202 Accepted in 11.13 ms  (job_id=0d881e63)
POST /documents  llama2-research-paper.pdf                            -> 202 Accepted in 91.15 ms  (job_id=27b00fe9)
POST /documents  vendor_msa_amendment.pdf                             -> 202 Accepted in  5.05 ms  (job_id=f286442c)
POST /documents  incident_report_q3.pdf                               -> 202 Accepted in  2.79 ms  (job_id=e0161c2a)
POST /documents  support_sla_addendum.pdf                             -> 202 Accepted in  3.17 ms  (job_id=d85521db)

All 5 uploads accepted. Processing is now happening in the background —
continue to Step 8 to watch it happen live.


## Step 8 — Poll status live until every job reaches `ready`

This is what the frontend does against `GET /documents/{id}` (or a
websocket) instead of blocking on the upload call: poll, redraw, repeat. The
loop below is bounded (it stops once every job is terminal or after a
timeout) so this cell always finishes on its own.

In [9]:
def render_status_table(job_ids: list[str]):
    rows = [{"job_id": jid, "filename": JOBS[jid].filename, "status": JOBS[jid].status.value,
              "attempts": JOBS[jid].attempt_count} for jid in job_ids]
    return pd.DataFrame(rows)

deadline = time.time() + 15
while time.time() < deadline:
    clear_output(wait=True)
    display(render_status_table(uploaded_ids))
    if all(JOBS[jid].status in TERMINAL for jid in uploaded_ids):
        print("All jobs reached a terminal state.")
        break
    time.sleep(0.2)
else:
    print("Timed out waiting for all jobs — this shouldn't happen for the non-failing demo docs above.")

,job_id,filename,status,attempts
0,0d881e63,Client_Contracts_Policies_and_Incident_Records...,ready,0
1,27b00fe9,llama2-research-paper.pdf,ready,0
2,f286442c,vendor_msa_amendment.pdf,ready,0
3,e0161c2a,incident_report_q3.pdf,ready,0
4,d85521db,support_sla_addendum.pdf,ready,0


All jobs reached a terminal state.


## Step 9 — Failure injection: retry, backoff, then dead-letter

Now upload one more document flagged `always_fail=True` — a stand-in for a
corrupt PDF that crashes the parser every time. Watch it retry with growing
backoff and land in `dead_letter` after `max_attempts`, with the error
preserved instead of the job silently vanishing.

In [10]:
poison_pill_id = upload_document("corrupted_scan.pdf", b"not actually a valid pdf", always_fail=True)
print("Uploaded poison-pill document:", poison_pill_id)

deadline = time.time() + 10
while time.time() < deadline:
    clear_output(wait=True)
    display(render_status_table([poison_pill_id]))
    if JOBS[poison_pill_id].status in TERMINAL:
        break
    time.sleep(0.2)

job = JOBS[poison_pill_id]
print(f"Final status: {job.status.value}")
print(f"Attempts:     {job.attempt_count}")
print(f"Error:        {job.error_message}")
print()
print("Full history:")
for ts, frm, to in job.history:
    print(f"  {ts}  {frm:10s} -> {to}")

,job_id,filename,status,attempts
0,a747ae96,corrupted_scan.pdf,dead_letter,3


Final status: dead_letter
Attempts:     3
Error:        parsing (ComplexPDFParser: PyMuPDF/pdfplumber/Tesseract) failed: simulated corrupt/poison-pill document

Full history:
  02:42:26.743  uploaded   -> parsing
  02:42:26.893  parsing    -> failed
  02:42:26.893  failed     -> retrying
  02:42:27.094  retrying   -> parsing
  02:42:27.244  parsing    -> failed
  02:42:27.244  failed     -> retrying
  02:42:27.644  retrying   -> parsing
  02:42:27.795  parsing    -> failed
  02:42:27.795  failed     -> retrying
  02:42:27.795  retrying   -> dead_letter


## Step 10 — Shut the workers down cleanly

Good practice before ending a notebook session — signal the workers to stop
polling the queue (they're daemon threads, so this isn't strictly required
for the notebook to exit, but it's what a graceful worker shutdown looks
like in production too).

In [11]:
stop_event.set()
for w in workers:
    w.join(timeout=1)

print("Workers stopped. Final job count:", len(JOBS))
print(render_status_table(list(JOBS.keys())))

Workers stopped. Final job count: 7
     job_id                                           filename       status  \
0  c0794948                                 large_contract.pdf        ready   
1  0d881e63  Client_Contracts_Policies_and_Incident_Records...        ready   
2  27b00fe9                          llama2-research-paper.pdf        ready   
3  f286442c                           vendor_msa_amendment.pdf        ready   
4  e0161c2a                             incident_report_q3.pdf        ready   
5  d85521db                           support_sla_addendum.pdf        ready   
6  a747ae96                                 corrupted_scan.pdf  dead_letter   

   attempts  
0         0  
1         0  
2         0  
3         0  
4         0  
5         0  
6         3  


## Step 11 — What's simulated here vs. what's real in your V1 code

| Piece in this notebook | Stands in for | Real V1 file it maps to |
|---|---|---|
| `OBJECT_STORAGE` dict | S3 / GCS / Azure Blob | new — replaces `data/uploads/` local disk |
| `JOBS` dict | PostgreSQL `documents` + `ingestion_jobs` tables | new — V1 has no metadata DB |
| `JOB_QUEUE` (`queue.Queue`) | SQS / RabbitMQ / Celery+Redis | new — V1 has no queue, calls are synchronous |
| `worker_loop` / `handle_job` | a pool of container replicas | wraps the **existing** `ComplexPDFParser` (`src/parsing.py`) and `MultimodalDocumentIngestion` (`src/ingestion.py`) — that logic doesn't change, it just moves off the request path |
| `process_stage()` sleeps | the actual parse/chunk/embed work | replace with real calls, per the comment in Step 5 |
| Step 8 polling loop | `GET /documents/{id}` from the frontend | new — replaces the blocking Streamlit call in `ui/app.py` |
| retry/backoff/dead-letter | queue-level retry policy (e.g. SQS redrive policy) | new |

**Bottom line:** none of your existing parsing/embedding logic gets rewritten
— it gets *relocated* from "called synchronously inside the Streamlit
request" to "called by a worker that a queue handed a job to." Everything in
this notebook (Steps 2–6) is the scaffolding that makes that relocation
possible.

See `Multimodal_RAG_Production_Design.docx` Section 5 for the full write-up,
and `Multimodal_RAG_Production_Design.ipynb` Section 5 for a shorter,
single-threaded version of this same state machine.